# 동반질환 23개 추출 (NeSy-SMP 재현용)

MIMIC-IV discharge note → medspaCy → 동반질환 long/wide CSV.

원 논문 `extract_comorbidities.py` 와 **추출 규칙(TargetRule 26개)과 부정/가족력 필터는 동일**하다.
두 가지만 고쳤다:

1. 루프 안 `list.index()` 제거 → dict 조회 (원본은 O(n²) 라 5만 건이면 끝나지 않는다)
2. `nlp.pipe()` 배치 처리

---

### 실행 전 준비

구글 드라이브 **내 드라이브 최상단**에 두 파일을 업로드한다.

| 파일 | 크기 |
|---|---|
| `discharge_icu.csv.gz` | 226 MB |
| `patients.csv.gz` | 1.4 MB |

### 셀 순서

1 설치 → **런타임 재시작** → 2 마운트 → 3 설정 → 4 파이프라인 → 5 추출 → 6 wide

> 5번이 오래 걸린다. 중간 저장/이어하기가 되니 끊겨도 셀 5를 다시 실행하면 이어서 진행된다.


## 1. 설치  (약 2분)


In [ ]:
!pip install -q medspacy
print('설치 완료. 위에 RESTART 버튼이 보이면 재시작한 뒤 셀 2부터 실행한다.')


## 2. 드라이브 마운트

권한 승인 창이 뜨면 본인 계정으로 직접 승인한다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. 설정

경로가 다르면 여기만 고친다.


In [ ]:
DRIVE    = '/content/drive/MyDrive'
NOTES    = f'{DRIVE}/discharge_icu.csv.gz'
PATIENTS = f'{DRIVE}/patients.csv.gz'      # 없으면 age 공란
OUT      = f'{DRIVE}/comorbidities.csv'    # long
OUT_W    = f'{DRIVE}/comorbidities_wide.csv'

CHECKPOINT_EVERY = 2000   # 중간 저장 간격(건)
BATCH_SIZE       = 20     # nlp.pipe 배치
LIMIT            = None   # 시험이면 200, 전체면 None

import os
for p, n in [(NOTES, 'discharge_icu.csv.gz'), (PATIENTS, 'patients.csv.gz')]:
    print(('  OK   ' if os.path.exists(p) else '  없음 ') + n)


## 4. 파이프라인 + 노트 로드


In [ ]:
import time, os
import pandas as pd
import medspacy
from medspacy.ner import TargetRule

nlp = medspacy.load(medspacy_enable=[
    'medspacy_pyrush', 'medspacy_target_matcher',
    'medspacy_context', 'medspacy_sectionizer'])
print('pipes:', nlp.pipe_names)

# ---- 원 논문 extract_comorbidities.py 의 TargetRule 목록 그대로 ----
TERMS = [
    'cancer', 'pneumonia', 'cirrhosis', 'dementia', 'kidney disease',
    'kidney failure', 'leukemia', 'hypertension', 'HIV', 'COPD',
    'Chronic Obstructive Pulmonary Disease', 'diabetes', 'diabetes mellitus',
    'trauma', 'coronary artery disease', 'Coronary Artery Disease', 'cad',
    'heart failure', 'atrial fibrillation', 'acute kidney injury',
    'peptic ulcer disease', 'cerebrovascular accident', 'metastatic disease',
    'metastatic cancer', 'lymphoma', 'AIDS',
]
nlp.get_pipe('medspacy_target_matcher').add([TargetRule(t, 'PROBLEM') for t in TERMS])
print(f'TargetRule {len(TERMS)}개 등록')

notes = pd.read_csv(NOTES, compression='gzip', low_memory=False)
notes = notes[notes.hadm_id.notna()].copy()
notes['hadm_id'] = notes['hadm_id'].astype('int64')
if LIMIT: notes = notes.head(LIMIT)
print(f'노트 {len(notes):,}건 / {notes.hadm_id.nunique():,} hadm')

# 원본의 list.index() 를 대체하는 dict
hadm2subject = dict(zip(notes.hadm_id, notes.subject_id))
hadm2age = {}
try:
    pat = pd.read_csv(PATIENTS, compression='gzip', usecols=['subject_id', 'anchor_age'])
    s2a = dict(zip(pat.subject_id, pat.anchor_age))
    hadm2age = {h: s2a.get(s, '') for h, s in hadm2subject.items()}
    print('age 조인 완료')
except Exception as e:
    print(f'[주의] patients 로드 실패 ({type(e).__name__}) — age 공란')


## 5. 추출  (오래 걸림 · 이어하기 지원)

끊기면 이 셀만 다시 실행한다. 이미 처리한 hadm 은 건너뛴다.


In [ ]:
done = set()
if os.path.exists(OUT):
    try:
        prev = pd.read_csv(OUT, usecols=['hadm_id'])
        done = set(prev.hadm_id.astype('int64'))
        print(f'이어하기: 이미 {len(done):,} hadm 처리됨')
    except Exception as e:
        print(f'기존 파일 못 읽음 ({type(e).__name__}) — 처음부터')

todo  = notes[~notes.hadm_id.isin(done)]
print(f'이번에 처리할 노트: {len(todo):,}건')
texts = todo['text'].astype(str).tolist()
hadms = todo['hadm_id'].tolist()

mode  = 'a' if done else 'w'
t0 = time.time()
n_ent = 0
with open(OUT, mode, encoding='utf-8') as f:
    if not done:
        f.write('subject_id,hadm_id,age,comorbidity\n')
    for i, (hadm, doc) in enumerate(zip(hadms, nlp.pipe(texts, batch_size=BATCH_SIZE)), 1):
        # 원본과 동일: 부정(NEGATED_EXISTENCE)/가족력(FAMILY) modifier 제외
        ents = {tg.text.lower() for tg, mod in doc._.context_graph.edges
                if mod.rule.category not in ('NEGATED_EXISTENCE', 'FAMILY')}
        sid = hadm2subject[hadm]
        age = hadm2age.get(hadm, '')
        for e in ents:
            if   e == 'chronic obstructive pulmonary disease': e = 'copd'
            elif e == 'coronary artery disease':               e = 'cad'
            f.write(f'{sid},{hadm},{age},{e}\n')
            n_ent += 1
        if i % CHECKPOINT_EVERY == 0:
            f.flush(); os.fsync(f.fileno())
            el = time.time() - t0; rem = (len(texts) - i) * el / i
            print(f'  {i:,}/{len(texts):,}  경과 {el/60:.1f}분  '
                  f'남은예상 {rem/60:.0f}분  누적 엔티티 {n_ent:,}', flush=True)
print(f'완료 {(time.time()-t0)/60:.1f}분 · 엔티티 {n_ent:,}개 -> {OUT}')


## 6. long → wide + 확인


In [ ]:
long = pd.read_csv(OUT)
print(f'long {len(long):,}행 / {long.hadm_id.nunique():,} hadm')
print(f'추출된 고유 동반질환 {long.comorbidity.nunique()}종')

wide = (long.assign(v=1)
            .pivot_table(index=['subject_id', 'hadm_id'], columns='comorbidity',
                         values='v', aggfunc='max', fill_value=0)
            .reset_index())
wide.to_csv(OUT_W, index=False)
print(f'wide {wide.shape} -> {OUT_W}')

cols = [c for c in wide.columns if c not in ('subject_id', 'hadm_id')]
prev = (100 * wide[cols].mean()).sort_values(ascending=False).round(1)
print(); print('동반질환별 유병률 (%):'); print(prev.to_string())
print(); print(f'동반질환이 하나도 없는 hadm: {100*(wide[cols].sum(axis=1)==0).mean():.1f}%')


---
## 결과 확인 포인트

- **23개가 다 나왔는지.** 목록 밖 항목이 나오거나 빠지면 TargetRule 매칭 문제다.
- **유병률이 상식적인지.** hypertension 은 ICU 환자에서 50% 안팎이 정상. 5% 라면 매칭 실패다.
- **결측 편향.** 노트가 없는 환자(코호트의 27.2%)는 여기 아예 안 들어온다.
  그 환자들이 더 위중하다 (섬망 26.1% vs 21.1%, 사망 9.1% vs 6.7%).
  wide 를 모델에 붙일 때 결측을 0으로 채우면 **위중한 쪽이 체계적으로 0** 이 된다. 보고서에 명시할 것.

## 결과 파일

`comorbidities.csv` (long) / `comorbidities_wide.csv` (wide) — 드라이브에 저장.
저장소 `NeSy-SMP/data/merge_comorbidities.py` 가 이 형식을 받는다.
